# Notebook 3: Embeddings y Búsqueda Semántica con AWS Bedrock

En este notebook aprenderás a:
- Generar embeddings de texto con Amazon Titan Embeddings
- Entender qué es un embedding y cómo se representa
- Calcular similitud semántica entre textos
- Construir un buscador semántico con FAISS
- Comparar búsqueda semántica vs búsqueda por palabras clave

---

> **Requisito:** boto3 instalado y credenciales AWS configuradas.

## ¿Por qué existe este notebook?

En el Notebook 2 aprendiste a formular preguntas al modelo. Pero hay un problema fundamental que aún no está resuelto: **el modelo no sabe nada de tus documentos**.

Si tienes cien páginas de documentación interna y alguien pregunta algo sobre ellas, no puedes meterle todo el texto al modelo en cada pregunta. Los modelos tienen un límite de cuánto texto pueden procesar a la vez, y además sería lento y caro.

La solución: convertir cada documento en una "huella numérica" de su significado (un **embedding**), guardar todas esas huellas en un índice, y cuando llegue una pregunta, buscar qué huellas se parecen más.

Este notebook construye ese motor de búsqueda. El Notebook 4 lo integra con el modelo para crear el sistema completo.

## 1. Instalación de dependencias

Las librerías nuevas en este notebook:

| Librería | Para qué sirve |
|----------|----------------|
| `numpy` | Operaciones matemáticas con vectores (listas de números) |
| `faiss-cpu` | Motor de búsqueda de vectores similares, creado por Meta/Facebook |

In [ ]:
# !pip install boto3 faiss-cpu numpy --quiet

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION=os.getenv("AWS_DEFAULT_REGION")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

## 2. Setup del cliente

Se crean dos constantes de modelo:
- `EMBEDDING_MODEL`: el modelo que convierte texto en vectores (Amazon Titan Embeddings)
- `GENERACION_MODEL`: Claude, que se usará para generar respuestas en el Notebook 4

Por ahora solo usamos el de embeddings.

In [ ]:
import boto3
import json
import numpy as np
import faiss
import math

REGION            = AWS_DEFAULT_REGION
EMBEDDING_MODEL   = "amazon.titan-embed-text-v2:0"
GENERACION_MODEL  = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

print("✅ Cliente Bedrock listo")
print(f"   Modelo de embeddings : {EMBEDDING_MODEL}")
print(f"   Modelo de generación : {GENERACION_MODEL}")

## 3. ¿Qué es un Embedding?

Un **embedding** es una representación numérica de un texto que captura su significado.

### La analogía del mapa

Imagina un mapa donde cada texto tiene una ubicación. Los textos que hablan de cosas similares quedan cerca. Los que no tienen relación quedan lejos.

Un embedding es exactamente eso: las coordenadas de un texto en ese mapa. Pero en lugar de 2 dimensiones (latitud y longitud), el mapa tiene **1536 dimensiones**.

No puedes visualizarlo, pero sí calcular matemáticamente qué tan cerca están dos puntos.

```
"El gato duerme"   → [0.12, -0.45, 0.78, ...] (1536 números)
"El felino reposa" → [0.11, -0.44, 0.79, ...] (muy cercano!)
"Receta de pizza"  → [0.89,  0.23, -0.54, ...] (muy lejano)
```

Amazon Titan Embeddings genera vectores de **1536 dimensiones** para cada texto que le envíes.

## 4. Generar el primer embedding

La función `generar_embedding` envía un texto al modelo Amazon Titan Embeddings y recibe su vector numérico.

### ¿Qué entra y qué sale?

**Entra:** un texto (string)  
**Sale:** un array numpy con 1536 números en formato float32

El campo `"inputText"` es el único que necesita Amazon Titan Embeddings en el body, mucho más simple que Claude.

In [ ]:
def generar_embedding(texto: str) -> np.ndarray:
    """Genera un embedding para un texto usando Amazon Titan Embeddings."""
    body = {"inputText": texto}
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return np.array(resultado["embedding"], dtype=np.float32)


# Generar embedding de ejemplo
texto_ejemplo = "AWS Bedrock permite usar modelos de lenguaje en la nube."
embedding = generar_embedding(texto_ejemplo)

print(f"Texto: '{texto_ejemplo}'")
print(f"Dimensiones del vector: {embedding.shape}")
print(f"Primeros 10 valores   : {embedding[:10].round(4)}")
print(f"Norma del vector      : {np.linalg.norm(embedding):.4f}")

## 5. Similitud semántica con Cosine Similarity

### ¿Cómo medimos "qué tan parecidos" son dos textos?

Con la **similitud coseno**: una fórmula matemática que mide el ángulo entre dos vectores. Si apuntan en la misma dirección, el ángulo es 0 y la similitud es 1.0 (idénticos). Si apuntan en direcciones opuestas, la similitud es -1.0.

| Valor | Significado |
|-------|-------------|
| `1.0` | Textos idénticos en significado |
| `0.7-0.9` | Muy relacionados |
| `0.3-0.6` | Algo relacionados |
| `< 0.3` | Sin relación aparente |
| `-1.0` | Significado opuesto |

En este bloque se calcula la similitud entre una frase de referencia sobre IA y cinco candidatos, incluyendo algunos completamente ajenos al tema.

**Entra:** dos vectores numpy  
**Sale:** un número entre -1 y 1

In [ ]:
def similitud_coseno(v1: np.ndarray, v2: np.ndarray) -> float:
    """Calcula la similitud coseno entre dos vectores."""
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


# Texto de referencia
referencia = "El aprendizaje automático es una rama de la inteligencia artificial."

# Textos a comparar
candidatos = [
    "Machine learning es un subcampo de la IA.",
    "Los algoritmos de ML aprenden de los datos.",
    "La inteligencia artificial transforma los negocios.",
    "El fútbol es el deporte más popular del mundo.",
    "La receta de la paella valenciana lleva arroz y mariscos.",
]

emb_referencia = generar_embedding(referencia)

print(f"Referencia: '{referencia}'")
print("\nSimilitud con candidatos:")
print("-" * 70)

resultados = []
for texto in candidatos:
    emb = generar_embedding(texto)
    sim = similitud_coseno(emb_referencia, emb)
    resultados.append((sim, texto))

resultados.sort(reverse=True)
for sim, texto in resultados:
    barra = "█" * int(sim * 30)
    print(f"{sim:.4f} {barra:<30} {texto}")

## 6. Construir un índice vectorial con FAISS

### ¿Qué es FAISS?

FAISS (Facebook AI Similarity Search) es una librería que permite buscar vectores similares de forma muy eficiente. En lugar de comparar tu consulta con cada documento uno por uno (lo cual sería muy lento con miles de documentos), FAISS organiza los vectores de forma inteligente para encontrar los más parecidos en milisegundos.

En producción se usaría un servicio gestionado (como Amazon OpenSearch), pero FAISS es perfecto para prototipar.

### La base de conocimiento

Los 14 documentos siguientes son los que se van a indexar. Están organizados en cuatro categorías temáticas: nube, IA/ML, bases de datos y seguridad. Más adelante veremos si el índice es capaz de agruparlos correctamente por significado.

**Entra:** 14 textos cortos  
**Sale:** (después de los siguientes pasos) un índice buscable

In [ ]:
# Base de conocimiento: artículos sobre tecnología
documentos = [
    # Nube
    "AWS ofrece más de 200 servicios en la nube, desde cómputo hasta IA.",
    "Amazon S3 es un servicio de almacenamiento de objetos altamente escalable.",
    "EC2 permite lanzar servidores virtuales en cuestión de segundos.",
    "Lambda es el servicio de computación serverless de AWS.",
    # IA / ML
    "Los modelos de lenguaje grande (LLM) son la base de la IA generativa actual.",
    "AWS Bedrock facilita el acceso a modelos de IA sin gestionar infraestructura.",
    "Los embeddings representan texto como vectores numéricos para búsqueda semántica.",
    "SageMaker es la plataforma de machine learning gestionada de AWS.",
    # Bases de datos
    "RDS permite ejecutar bases de datos relacionales como PostgreSQL y MySQL en AWS.",
    "DynamoDB es una base de datos NoSQL totalmente gestionada y de baja latencia.",
    "Amazon Aurora es compatible con MySQL y PostgreSQL y es hasta 5x más rápida.",
    # Seguridad
    "IAM gestiona el acceso a recursos de AWS mediante roles, usuarios y políticas.",
    "AWS KMS permite crear y gestionar claves de cifrado de forma centralizada.",
    "AWS Shield protege contra ataques de denegación de servicio (DDoS).",
]

print(f"Total de documentos en la base de conocimiento: {len(documentos)}")

### Generar embeddings para todos los documentos

Aquí se llama a `generar_embedding` para cada uno de los 14 documentos. El resultado es una lista de 14 vectores de 1536 dimensiones.

Después se apilan en una matriz (una tabla de números: 14 filas × 1536 columnas) con `np.vstack`.

**Entra:** 14 textos  
**Sale:** una matriz numpy de forma (14, 1536)

In [ ]:
# Generar embeddings para todos los documentos
print("Generando embeddings para los documentos...")
embeddings_docs = []

for i, doc in enumerate(documentos):
    emb = generar_embedding(doc)
    embeddings_docs.append(emb)
    print(f"  [{i+1:2}/{len(documentos)}] ✓ {doc[:60]}..." if len(doc) > 60 else f"  [{i+1:2}/{len(documentos)}] ✓ {doc}")

# Crear matriz de embeddings
matriz_embeddings = np.vstack(embeddings_docs)
dimension = matriz_embeddings.shape[1]

print(f"\n✅ Embeddings generados: {matriz_embeddings.shape}")

### Crear el índice FAISS

Dos pasos importantes aquí:

1. **`faiss.normalize_L2`**: ajusta cada vector para que su "longitud" matemática sea exactamente 1. Esto es necesario para que el producto interno (Inner Product) sea equivalente a la similitud coseno.

2. **`faiss.IndexFlatIP`**: crea el índice. "IP" = Inner Product. "Flat" = sin compresión, exacto. Es el más simple y preciso.

**Entra:** la matriz de embeddings normalizados  
**Sale:** un índice FAISS con 14 vectores indexados y listos para buscar

In [ ]:
# Construir índice FAISS (Inner Product = Cosine Similarity con vectores normalizados)
faiss.normalize_L2(matriz_embeddings)  # Normalizar para usar producto interno como coseno

indice = faiss.IndexFlatIP(dimension)  # IP = Inner Product
indice.add(matriz_embeddings)

print(f"✅ Índice FAISS creado")
print(f"   Vectores indexados : {indice.ntotal}")
print(f"   Dimensión          : {dimension}")

## 7. Función de búsqueda semántica

### Cómo funciona la búsqueda

1. Se convierte la consulta del usuario en un embedding (igual que se hizo con los documentos)
2. Se normaliza ese vector
3. Se busca en el índice FAISS los `top_k` vectores más similares
4. Se devuelven los documentos correspondientes con su puntuación de similitud

**Entra:** texto de la consulta + número de resultados deseados  
**Sale:** lista de documentos ordenados por relevancia, con su score

El score es un número entre 0 y 1. Cuanto más cercano a 1, más relevante es el documento para la consulta.

In [ ]:
def buscar_semantico(query: str, top_k: int = 3) -> list[dict]:
    """Busca los documentos más relevantes para una consulta."""
    # Generar embedding de la consulta
    emb_query = generar_embedding(query).reshape(1, -1)
    faiss.normalize_L2(emb_query)

    # Buscar en el índice
    scores, indices = indice.search(emb_query, top_k)

    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        resultados.append({
            "documento": documentos[idx],
            "score": float(score),
            "indice": int(idx)
        })
    return resultados


# Prueba 1: Consulta sobre IA
query = "¿Cómo puedo usar inteligencia artificial en AWS?"
print(f"🔍 Consulta: '{query}'")
print("\nResultados más relevantes:")
print("-" * 70)
for i, r in enumerate(buscar_semantico(query, top_k=4), 1):
    print(f"{i}. [Score: {r['score']:.4f}] {r['documento']}")

In [ ]:
# Prueba 2: Consulta sobre bases de datos
query2 = "¿Qué opciones de almacenamiento de datos existen?"
print(f"🔍 Consulta: '{query2}'")
print("\nResultados más relevantes:")
print("-" * 70)
for i, r in enumerate(buscar_semantico(query2, top_k=4), 1):
    print(f"{i}. [Score: {r['score']:.4f}] {r['documento']}")

In [ ]:
# Prueba 3: Consulta sobre seguridad
query3 = "¿Cómo puedo proteger mi infraestructura de ataques?"
print(f"🔍 Consulta: '{query3}'")
print("\nResultados más relevantes:")
print("-" * 70)
for i, r in enumerate(buscar_semantico(query3, top_k=3), 1):
    print(f"{i}. [Score: {r['score']:.4f}] {r['documento']}")

## 8. Búsqueda semántica vs Búsqueda por palabras clave

### El problema de la búsqueda por keywords

La búsqueda tradicional busca coincidencias de palabras exactas. Si el usuario dice "funciones sin servidor" pero el documento habla de "Lambda" y "serverless", no hay match.

La búsqueda semántica entiende que "computación sin servidor" y "serverless" son lo mismo.

### El experimento

Se usa la misma consulta con ambos métodos. La búsqueda por keywords se hace manualmente: cuenta cuántas palabras de la consulta aparecen en el documento (ignorando palabras comunes como "de", "la", "el").

**Entra:** consulta con vocabulario diferente al de los documentos  
**Sale:** los resultados de cada método — verás que el semántico encuentra el documento correcto aunque no comparte palabras

In [ ]:
def busqueda_keywords(query: str, documentos: list[str], top_k: int = 3) -> list[dict]:
    """Búsqueda simple por coincidencia de palabras clave."""
    palabras_query = set(query.lower().split())
    # Eliminar stopwords básicas
    stopwords = {"de", "la", "el", "en", "y", "a", "los", "las", "un", "una", "es", "que", "con"}
    palabras_query -= stopwords

    scores = []
    for i, doc in enumerate(documentos):
        palabras_doc = set(doc.lower().split()) - stopwords
        coincidencias = len(palabras_query & palabras_doc)
        scores.append({"documento": doc, "score": coincidencias, "indice": i})

    return sorted(scores, key=lambda x: x["score"], reverse=True)[:top_k]


# Consulta con vocabulario diferente al de los documentos
# Los documentos hablan de "Lambda" y "serverless", el usuario dice "funciones sin servidor"
query_test = "¿Qué es la computación sin servidor?"

print(f"Consulta: '{query_test}'")
print("\n--- BÚSQUEDA POR KEYWORDS ---")
kw_results = busqueda_keywords(query_test, documentos, top_k=3)
for i, r in enumerate(kw_results, 1):
    print(f"{i}. [Coincidencias: {r['score']}] {r['documento']}")

print("\n--- BÚSQUEDA SEMÁNTICA ---")
sem_results = buscar_semantico(query_test, top_k=3)
for i, r in enumerate(sem_results, 1):
    print(f"{i}. [Score: {r['score']:.4f}] {r['documento']}")

## 9. Persistir y cargar el índice

### ¿Por qué guardar el índice en disco?

Generar los embeddings tiene un coste: tiempo y llamadas a la API. En producción, los documentos no cambian con cada consulta. Lo más eficiente es:

1. Generar embeddings **una vez** y guardar el índice
2. En cada consulta, **cargar el índice** ya construido y buscar en él

Este bloque guarda dos archivos:
- `bedrock_index.faiss`: el índice vectorial
- `documentos.pkl`: la lista de textos originales (necesaria para mostrar el resultado)

**Entra:** el índice FAISS en memoria  
**Sale:** dos archivos en disco

In [ ]:
import os
import pickle

# Guardar índice y documentos
faiss.write_index(indice, "bedrock_index.faiss")

with open("documentos.pkl", "wb") as f:
    pickle.dump(documentos, f)

print(f"✅ Índice guardado: bedrock_index.faiss ({os.path.getsize('bedrock_index.faiss')} bytes)")
print(f"✅ Documentos guardados: documentos.pkl")

### Cargar el índice desde disco

Aquí se simula reiniciar la aplicación: se carga el índice desde disco y se verifica que sigue funcionando igual.

**Entra:** los archivos en disco  
**Sale:** el índice y los documentos cargados en memoria, listos para buscar

In [ ]:
# Cargar índice desde disco (simula reiniciar la aplicación)
indice_cargado = faiss.read_index("bedrock_index.faiss")

with open("documentos.pkl", "rb") as f:
    documentos_cargados = pickle.load(f)

print(f"✅ Índice cargado con {indice_cargado.ntotal} vectores")
print(f"✅ Documentos cargados: {len(documentos_cargados)}")

# Verificar que funciona igual
query_verificacion = "seguridad en AWS"
emb = generar_embedding(query_verificacion).reshape(1, -1)
faiss.normalize_L2(emb)
scores, indices = indice_cargado.search(emb, 2)

print(f"\nVerificación — Consulta: '{query_verificacion}'")
for score, idx in zip(scores[0], indices[0]):
    print(f"  [{score:.4f}] {documentos_cargados[idx]}")

## 10. Visualizar similitudes en 2D (PCA)

### ¿Qué es PCA y por qué se usa aquí?

Los embeddings tienen 1536 dimensiones. Los humanos solo podemos visualizar 2 o 3. **PCA** (Análisis de Componentes Principales) es una técnica matemática que reduce esas 1536 dimensiones a 2, intentando conservar al máximo las distancias entre puntos.

El resultado no es perfecto, pero da una idea visual de cómo se agrupan los documentos por significado.

Si la visualización muestra los documentos del mismo tema juntos (nube, IA/ML, BBDD, seguridad), significa que los embeddings están capturando correctamente la semántica.

**Entra:** la matriz de embeddings (14 × 1536)  
**Sale:** un gráfico 2D donde cada punto es un documento

In [ ]:
try:
    import matplotlib.pyplot as plt
    from sklearn.decomposition import PCA

    # Etiquetas y colores por categoría
    categorias = (
        ["Nube"] * 4 +
        ["IA/ML"] * 4 +
        ["BBDD"] * 3 +
        ["Seguridad"] * 3
    )
    colores_map = {"Nube": "#4A90D9", "IA/ML": "#E74C3C", "BBDD": "#2ECC71", "Seguridad": "#F39C12"}

    # Reducción PCA a 2D
    pca = PCA(n_components=2)
    coords = pca.fit_transform(matriz_embeddings)

    # Gráfico
    fig, ax = plt.subplots(figsize=(12, 8))
    for i, (x, y) in enumerate(coords):
        cat = categorias[i]
        color = colores_map[cat]
        ax.scatter(x, y, c=color, s=100, zorder=3)
        label = documentos[i][:40] + "..."
        ax.annotate(label, (x, y), fontsize=7, ha="center", va="bottom",
                    xytext=(0, 8), textcoords="offset points")

    # Leyenda
    for cat, color in colores_map.items():
        ax.scatter([], [], c=color, label=cat, s=80)
    ax.legend(title="Categoría", loc="upper right")

    ax.set_title("Visualización 2D de Embeddings (PCA)\nDocumentos agrupados por similitud semántica", fontsize=13)
    ax.set_xlabel("Componente principal 1")
    ax.set_ylabel("Componente principal 2")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("embeddings_2d.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Gráfico guardado como embeddings_2d.png")

except ImportError:
    print("Para visualización instala: pip install matplotlib scikit-learn")

## 11. Resumen y próximos pasos

En este notebook aprendiste a:

✅ Generar embeddings con **Amazon Titan Embeddings** vía Bedrock  
✅ Calcular **similitud coseno** entre textos  
✅ Construir un **índice vectorial con FAISS**  
✅ Realizar búsquedas semánticas eficientes  
✅ Comparar búsqueda semántica vs keywords  
✅ Persistir y cargar índices vectoriales  
✅ Visualizar embeddings con PCA  

### ¿Para qué sirve esto en producción?

| Caso de uso | Descripción |
|-------------|-------------|
| **RAG** | Recuperar contexto relevante para enriquecer prompts |
| **Buscadores** | Búsqueda semántica en documentación o catálogos |
| **Recomendaciones** | Encontrar contenido similar al que el usuario consume |
| **Deduplicación** | Detectar documentos casi idénticos en grandes colecciones |

---

**Siguiente notebook →** RAG (Retrieval-Augmented Generation) básico con Bedrock